# Disney Box Office: What Drives Revenue?

**Dataset:** 579 Disney films (Kaggle "Disney Movies" dataset) with release date, genre, MPAA rating,
nominal gross (`total_gross`) and inflation-adjusted gross (`inflation_adjusted_gross`).

**Questions**

1. How is box-office revenue distributed, and is the "average" film a misleading summary?
2. Which genres and MPAA ratings earn the most, in total and for a *typical* film?
3. Do the differences between genres hold up statistically, or are they noise?
4. How does the picture change once revenue is adjusted for inflation?

**How to run:** put `disney_movies.csv` in the repo's `data/` folder (see `data/README.md`), install
`requirements.txt`, then *Run All*. Every number quoted in the findings section is computed from the data
by the notebook, not typed in.

## 1. Setup and data loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
MIN_N = 10   # minimum films for a group to be compared / tested


def money(x, pos=None):
    """Format a dollar amount compactly: $12M, $1.2B."""
    if pd.isna(x):
        return ""
    return f"${x / 1e9:,.2f}B" if abs(x) >= 1e9 else f"${x / 1e6:,.0f}M"


def money_axis(x, pos=None):
    """Compact tick labels: $0, $20M, $2.5B."""
    if x == 0:
        return "$0"
    return f"${x / 1e9:g}B" if abs(x) >= 1e9 else f"${x / 1e6:g}M"

In [ ]:
CANDIDATES = [Path("data/disney_movies.csv"), Path("../data/disney_movies.csv")]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "disney_movies.csv not found. Place it in the repo's data/ folder (see data/README.md)."
    )

raw = pd.read_csv(DATA_PATH)
EXPECTED = {"movie_title", "release_date", "genre", "mpaa_rating", "total_gross", "inflation_adjusted_gross"}
missing_cols = EXPECTED - set(raw.columns)
assert not missing_cols, f"Missing expected columns: {missing_cols}"
print(f"Loaded {len(raw):,} rows x {raw.shape[1]} columns from {DATA_PATH}")
raw.head()

## 2. Data-quality audit

Before any analysis: what is missing, what is duplicated, and which values look wrong?

In [ ]:
audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(1),
})
display(audit)

zero_mask = (raw["total_gross"] == 0) | (raw["inflation_adjusted_gross"] == 0)
print(f"Fully duplicated rows : {raw.duplicated().sum()}")
print(f"Repeated movie titles : {raw['movie_title'].duplicated().sum()}")
print(f"Rows with a 0 gross   : {zero_mask.sum()}  (treated as 'not reported' and excluded from revenue statistics)")
raw.loc[zero_mask]

## 3. Cleaning

Decisions, and why:

- **Missing genre / rating** become an explicit `Unknown` category instead of the made-up label
  `not information`, so they stay visible but are easy to exclude from comparisons.
- **A gross of `0`** is not a real box-office result for a released film, so it is set to missing and excluded
  from revenue statistics (the count is reported above).
- **Release date** is parsed to a real date, and `year` / `decade` are derived for the time analysis.
- **Inflation factor** (`inflation_adjusted_gross / total_gross`) is added: it measures how much a film's
  gross is scaled up to today's dollars, and shows why nominal grosses from different eras are not comparable.

In [ ]:
df = raw.copy()
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"] = df["release_date"].dt.year
df["decade"] = (df["year"] // 10 * 10).astype("Int64")

for col in ["genre", "mpaa_rating"]:
    df[col] = df[col].fillna("Unknown").str.strip()

for col in ["total_gross", "inflation_adjusted_gross"]:
    df[col] = df[col].where(df[col] > 0)

df["inflation_factor"] = df["inflation_adjusted_gross"] / df["total_gross"]

rev = df.dropna(subset=["total_gross", "inflation_adjusted_gross"]).copy()
print(f"Films used for revenue analysis: {len(rev):,} of {len(df):,}")
print(f"Release years: {int(rev['year'].min())} - {int(rev['year'].max())}")

## 4. How is revenue distributed?

Box-office revenue is extremely right-skewed: a handful of blockbusters dwarf everything else. When that
happens the **mean** is dragged upward, so the **median** is the more honest "typical film". The histograms use a
log scale so the whole range is visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col, title in zip(
    axes,
    ["total_gross", "inflation_adjusted_gross"],
    ["Nominal gross", "Inflation-adjusted gross"],
):
    s = rev[col]
    bins = np.logspace(np.log10(s.min()), np.log10(s.max()), 30)
    sns.histplot(s, bins=bins, ax=ax, color="tab:blue")
    ax.set_xscale("log")
    ax.axvline(s.median(), color="tab:red", ls="--", label=f"median {money(s.median())}")
    ax.axvline(s.mean(), color="tab:orange", ls=":", lw=2, label=f"mean {money(s.mean())}")
    ax.xaxis.set_major_formatter(money_axis)
    ax.set_title(title)
    ax.set_xlabel("Gross (log scale)")
    ax.set_ylabel("Number of films")
    ax.legend()
fig.suptitle("Revenue is heavily right-skewed: the mean overstates the typical film", y=1.02, fontweight="bold")
plt.tight_layout()
plt.show()

skew = rev[["total_gross", "inflation_adjusted_gross"]].skew()
print("Skewness  nominal: {:.1f}   inflation-adjusted: {:.1f}".format(*skew))

## 5. Genre analysis

Total revenue answers *"where did the money come from?"*; the median answers *"what does a typical film in
this genre earn?"*. They can tell different stories, so both are shown. Medians are only plotted for genres with
at least `MIN_N` films, because a median of 2-3 films is not meaningful.

In [ ]:
def summarize(frame, by):
    out = frame.groupby(by, observed=True).agg(
        films=("movie_title", "count"),
        total_adj=("inflation_adjusted_gross", "sum"),
        median_adj=("inflation_adjusted_gross", "median"),
        mean_adj=("inflation_adjusted_gross", "mean"),
        median_nominal=("total_gross", "median"),
    )
    out["share_of_total_%"] = out["total_adj"] / out["total_adj"].sum() * 100
    return out.sort_values("total_adj", ascending=False)


genre_tbl = summarize(rev, "genre")

shown = genre_tbl.copy()
for c in ["total_adj", "median_adj", "mean_adj", "median_nominal"]:
    shown[c] = shown[c].map(money)
shown["share_of_total_%"] = shown["share_of_total_%"].round(1)
shown

In [ ]:
named = genre_tbl.drop(index="Unknown", errors="ignore")
eligible = named[named["films"] >= MIN_N]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

t = named["total_adj"].sort_values()
axes[0].barh(t.index, t.values, color="tab:blue")
axes[0].xaxis.set_major_formatter(money_axis)
axes[0].set_title("Total inflation-adjusted gross by genre")
axes[0].set_xlabel("Total gross")

m = eligible["median_adj"].sort_values()
axes[1].barh(m.index, m.values, color="tab:green")
for y, (name, val) in enumerate(m.items()):
    axes[1].text(val, y, f"  n={int(eligible.loc[name, 'films'])}", va="center", fontsize=9, color="dimgray")
axes[1].xaxis.set_major_formatter(money_axis)
axes[1].set_title(f"Median film, genres with at least {MIN_N} films")
axes[1].set_xlabel("Median gross per film")
axes[1].set_xlim(0, m.max() * 1.18)

plt.tight_layout()
plt.show()

In [ ]:
order = eligible["median_adj"].sort_values(ascending=False).index
plot_df = rev[rev["genre"].isin(order)]

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.boxplot(data=plot_df, y="genre", x="inflation_adjusted_gross", order=order,
            color="tab:blue", fliersize=3, ax=ax)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(money_axis)
ax.set_xlabel("Inflation-adjusted gross per film (log scale)")
ax.set_ylabel("")
ax.set_title("Spread of revenue within each genre")
plt.tight_layout()
plt.show()

### Are the genre differences real?

Revenue is not normally distributed, so a one-way ANOVA is inappropriate. The **Kruskal-Wallis** test compares
the genres without assuming normality (run on log revenue; only genres with at least `MIN_N` films). The
effect size (epsilon-squared) says how much of the variation genre explains, which matters more than the p-value.

In [ ]:
groups = [g["inflation_adjusted_gross"].values for _, g in plot_df.groupby("genre")]
H, p = stats.kruskal(*groups)
k, n_obs = len(groups), sum(len(g) for g in groups)
epsilon_sq = max(0.0, (H - k + 1) / (n_obs - k))

print(f"Kruskal-Wallis across {k} genres (n={n_obs}):  H = {H:.2f},  p = {p:.4g}")
print(f"Effect size (epsilon-squared) = {epsilon_sq:.3f}  ->  genre explains roughly {epsilon_sq:.0%} of the rank variation in revenue")

## 6. MPAA rating

In [ ]:
rating_tbl = summarize(rev, "mpaa_rating")
rating_named = rating_tbl.drop(index="Unknown", errors="ignore")
rating_eligible = rating_named[rating_named["films"] >= MIN_N]

shown = rating_tbl.copy()
for c in ["total_adj", "median_adj", "mean_adj", "median_nominal"]:
    shown[c] = shown[c].map(money)
shown["share_of_total_%"] = shown["share_of_total_%"].round(1)
display(shown)

r_order = rating_eligible["median_adj"].sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=rev[rev["mpaa_rating"].isin(r_order)], x="mpaa_rating", y="inflation_adjusted_gross",
            order=r_order, color="tab:purple", fliersize=3, ax=ax)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(money_axis)
ax.set_xlabel("MPAA rating")
ax.set_ylabel("Inflation-adjusted gross (log scale)")
ax.set_title(f"Revenue by rating (ratings with at least {MIN_N} films)")
plt.tight_layout()
plt.show()

r_groups = [g["inflation_adjusted_gross"].values for _, g in rev[rev["mpaa_rating"].isin(r_order)].groupby("mpaa_rating")]
Hr, pr = stats.kruskal(*r_groups)
print(f"Kruskal-Wallis across {len(r_groups)} ratings:  H = {Hr:.2f},  p = {pr:.4g}")

## 7. Revenue over time, and why inflation adjustment matters

Comparing a 1940 gross with a 2016 gross in nominal dollars is not a like-for-like comparison. The inflation factor
shows how heavily older films are scaled up; the right-hand chart shows how the "typical film" trend changes
depending on which measure is used.

In [ ]:
by_decade = rev.groupby("decade").agg(
    films=("movie_title", "count"),
    median_nominal=("total_gross", "median"),
    median_adjusted=("inflation_adjusted_gross", "median"),
    median_factor=("inflation_factor", "median"),
)
by_decade.index = by_decade.index.astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

axes[0].bar(by_decade.index.astype(str) + "s", by_decade["films"], color="tab:blue")
axes[0].set_title("Films released per decade")
axes[0].set_ylabel("Number of films")
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(by_decade.index.astype(str) + "s", by_decade["median_nominal"], marker="o", label="Nominal")
axes[1].plot(by_decade.index.astype(str) + "s", by_decade["median_adjusted"], marker="o", label="Inflation-adjusted")
axes[1].yaxis.set_major_formatter(money_axis)
axes[1].set_title("Median gross per film, by decade")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

shown = by_decade.copy()
shown["median_nominal"] = shown["median_nominal"].map(money)
shown["median_adjusted"] = shown["median_adjusted"].map(money)
shown["median_factor"] = shown["median_factor"].round(2)
shown

In [ ]:
top_genres = named.sort_values("films", ascending=False).head(5).index
decade_counts = (
    rev[rev["genre"].isin(top_genres)]
    .groupby(["decade", "genre"]).size().unstack(fill_value=0)
)
decade_counts.index = decade_counts.index.astype(int)
decade_counts = decade_counts[decade_counts.sum(axis=1) >= MIN_N]
decade_share = decade_counts.div(decade_counts.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(decade_share, annot=True, fmt=".0%", cmap="Blues", cbar_kws={"label": "Share of the decade's top-5-genre films"}, ax=ax)
ax.set_title("Genre mix by decade (top 5 genres; decades with at least 10 such films)")
ax.set_ylabel("Decade")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

rho_nom, p_nom = stats.spearmanr(rev["year"], rev["total_gross"])
rho_adj, p_adj = stats.spearmanr(rev["year"], rev["inflation_adjusted_gross"])
print(f"Spearman correlation, release year vs nominal gross           : rho = {rho_nom:+.2f} (p = {p_nom:.3g})")
print(f"Spearman correlation, release year vs inflation-adjusted gross: rho = {rho_adj:+.2f} (p = {p_adj:.3g})")

## 8. Top films: nominal vs inflation-adjusted

Ranking by nominal gross favours recent releases; ranking by inflation-adjusted gross shows which films were
the biggest hits relative to their era.

In [ ]:
rev["rank_nominal"] = rev["total_gross"].rank(ascending=False, method="min").astype(int)
rev["rank_adjusted"] = rev["inflation_adjusted_gross"].rank(ascending=False, method="min").astype(int)
rev["rank_gain"] = rev["rank_nominal"] - rev["rank_adjusted"]

cols = ["movie_title", "year", "genre", "total_gross", "inflation_adjusted_gross", "rank_nominal", "rank_adjusted"]
top10 = rev.nsmallest(10, "rank_adjusted")[cols]

fig, ax = plt.subplots(figsize=(10, 5))
labels = top10["movie_title"] + " (" + top10["year"].astype(int).astype(str) + ")"
ax.barh(labels[::-1], top10["inflation_adjusted_gross"][::-1], color="tab:blue", label="Inflation-adjusted")
ax.barh(labels[::-1], top10["total_gross"][::-1], color="tab:orange", height=0.4, label="Nominal")
ax.xaxis.set_major_formatter(money_axis)
ax.set_title("Top 10 films by inflation-adjusted gross")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

shown = top10.copy()
shown["total_gross"] = shown["total_gross"].map(money)
shown["inflation_adjusted_gross"] = shown["inflation_adjusted_gross"].map(money)
display(shown.reset_index(drop=True))

print("Biggest climbers when adjusting for inflation (rank_nominal -> rank_adjusted):")
movers = rev.nlargest(5, "rank_gain")[["movie_title", "year", "rank_nominal", "rank_adjusted", "rank_gain"]]
display(movers.reset_index(drop=True))

## 9. Key findings (computed from the data above)

In [ ]:
top_total = named.index[0]
top_median = eligible["median_adj"].idxmax()
low_median = eligible["median_adj"].idxmin()
best_title = top10.iloc[0]

print(f"1. Skew: the mean film grosses {money(rev['inflation_adjusted_gross'].mean())} but the median is only "
      f"{money(rev['inflation_adjusted_gross'].median())} (inflation-adjusted). A few blockbusters pull the average up.")
print(f"2. Most total revenue comes from {top_total} "
      f"({named.loc[top_total, 'share_of_total_%']:.0f}% of the inflation-adjusted total, {int(named.loc[top_total, 'films'])} films).")
print(f"3. The best *typical* film is in {top_median} (median {money(eligible.loc[top_median, 'median_adj'])}); "
      f"the weakest of the genres with at least {MIN_N} films is {low_median} (median {money(eligible.loc[low_median, 'median_adj'])}).")
verdict = "are statistically distinguishable" if p < 0.05 else "cannot be statistically distinguished"
if epsilon_sq < 0.06:
    reading = "so most of the spread is within genres, not between them"
elif epsilon_sq < 0.14:
    reading = "a moderate share: genre matters, but much of the spread is still within genres"
else:
    reading = "a large share: genre is a major driver of revenue differences"
print(f"4. Genre differences in revenue {verdict} (Kruskal-Wallis p = {p:.3g}); genre explains about "
      f"{epsilon_sq:.0%} of the rank variation, {reading}.")
print(f"5. Once adjusted for inflation, the all-time #1 is {best_title['movie_title']} ({int(best_title['year'])}) "
      f"at {money(best_title['inflation_adjusted_gross'])}, versus {money(best_title['total_gross'])} nominal.")

## 10. Limitations and next steps

**Limitations**

- Each film has a **single genre label**, so cross-genre films (e.g. animated musical comedies) are forced into one bucket.
- The dataset ends in **2016**, and the early decades contain few films, so decade comparisons for the 1930s-1950s are noisy.
- Inflation adjustment uses the dataset's own method, which is not documented; treat adjusted figures as approximate.
- There is **no budget data**, so this measures revenue, not profitability or return on investment.
- Differences between groups are **observational**: genre and rating are correlated with era, franchise and release strategy.

**Next steps**

- Join production-budget data to analyse profit and ROI rather than revenue alone.
- Add franchise / studio-brand flags (Marvel, Star Wars, Pixar) to separate franchise effects from genre effects.
- Model `log(inflation_adjusted_gross)` on genre, rating and decade to estimate effects while controlling for each other.